# Experimento 02 — Incident Management
**PROJENER.AI SL · UAX · Máster Big Data · 2026**  
Autora: Edurne Martínez de Contrasta

Replicación del framework multi-agente en gestión de incidencias IT (ACME Industrial S.A.).  
Mismos modelos y métricas que Experimento 01.

**Dataset:** 20 casos — Simple (C01-C08), Medio (C09-C14), Complejo (C15-C20)  
**6 casos requieren HiL** — todos Complejo: brecha GDPR, caída ERP, ransomware, pérdida BBDD, acceso nóminas, intrusión OT/SCADA  
**Agentes:** Reporter, Technical, Security, Operations, Resolver  
**Modelos:** M1 (RPA baseline), M2 (Single Agent 8b), M4 (Multi-Agent mixto)

In [14]:
# ============================================================
# PROJENER.AI SL — UAX Máster Big Data 2026
# Experimento 02: Incident Management Multi-Agent LLM
# Misma empresa: ACME Industrial S.A.
# ============================================================

import sys, re, time, json, os
sys.path.insert(0, r"C:\Users\edurn\practicas projener")
os.environ["GROQ_API_KEY"] = "MI_TOKEN_GROQ"
os.environ["OTEL_SDK_DISABLED"] = "true"

# Verificar setup
print(f"Python: {sys.version[:6]}")
print(f"Carpeta: {os.path.exists(r'C:\Users\edurn\practicas projener')}")
print(f"Dataset: {os.path.exists(r'C:\Users\edurn\practicas projener\casos_incident_management.json')}")

Python: 3.12.1
Carpeta: True
Dataset: True


## Dataset y APIs Mock

In [15]:
# ============================================================
# APIS MOCK — Incident Management
# ============================================================

def get_incidente_info(incidente_id, categoria, sistema_afectado):
    """API mock — información técnica del incidente"""
    riesgo_por_categoria = {
        "seguridad": "alto", "datos": "alto",
        "infraestructura": "medio", "red": "medio",
        "software": "bajo", "hardware": "bajo", "acceso": "bajo"
    }
    return {
        "incidente_id": incidente_id,
        "categoria": categoria,
        "sistema": sistema_afectado,
        "riesgo_tecnico": riesgo_por_categoria.get(categoria, "medio"),
        "requiere_especialista": categoria in ["seguridad", "datos"]
    }

def verificar_impacto_operativo(departamento, impacto_usuarios, sistema_critico):
    """API mock — impacto en operaciones"""
    nivel = "critico" if sistema_critico and impacto_usuarios > 100 else \
            "alto" if impacto_usuarios > 50 or sistema_critico else \
            "medio" if impacto_usuarios > 10 else "bajo"
    return {
        "departamento": departamento,
        "usuarios_afectados": impacto_usuarios,
        "sistema_critico": sistema_critico,
        "nivel_impacto": nivel,
        "requiere_escalacion": nivel in ["critico", "alto"]
    }

def verificar_riesgo_seguridad(incidente_seguridad, datos_en_riesgo, categoria):
    """API mock — evaluación de riesgo de seguridad"""
    es_critico = incidente_seguridad and datos_en_riesgo
    return {
        "incidente_seguridad": incidente_seguridad,
        "datos_en_riesgo": datos_en_riesgo,
        "requiere_notificacion_gdpr": datos_en_riesgo and incidente_seguridad,
        "nivel_riesgo_seguridad": "critico" if es_critico else "alto" if incidente_seguridad or datos_en_riesgo else "bajo",
        "accion_recomendada": "escalar_inmediato" if es_critico else "revisar" if incidente_seguridad else "monitorizar"
    }

def registrar_incidente(caso):
    """API mock — registro del incidente"""
    return {
        "ticket_id": f"INC-2026-{caso['id']}",
        "estado": "abierto",
        "prioridad": caso.get("prioridad_reportada", "media"),
        "usuario": caso.get("usuario", ""),
        "descripcion": caso.get("descripcion", "")
    }

def cargar_casos_incident():
    """Cargar dataset de incident management"""
    with open(r"C:\Users\edurn\practicas projener\casos_incident_management.json", encoding="utf-8") as f:
        data = json.load(f)
    return data["casos"]

import json
casos_inc = cargar_casos_incident()
print(f"✅ {len(casos_inc)} casos cargados")
print(f"   Simple: {sum(1 for c in casos_inc if c['nivel']=='Simple')}")
print(f"   Medio: {sum(1 for c in casos_inc if c['nivel']=='Medio')}")
print(f"   Complejo: {sum(1 for c in casos_inc if c['nivel']=='Complejo')}")
print(f"   Requieren HiL: {sum(1 for c in casos_inc if c['ground_truth']['requiere_hil'])}")

✅ 20 casos cargados
   Simple: 8
   Medio: 6
   Complejo: 6
   Requieren HiL: 6


## M1 — Baseline RPA

In [16]:
# ============================================================
# M1 — BASELINE RPA (Incident Management)
# Reglas if-then sin LLM
# ============================================================

def procesar_incidente_m1_rpa(caso):
    """Baseline RPA — reglas if-then para incident management"""
    t_ini = time.time()
    
    # Reglas deterministas
    if caso.get("incidente_seguridad") and caso.get("datos_en_riesgo"):
        decision = "escalar_critico"
        escala = False  # RPA no escala al humano, solo clasifica
    elif caso.get("sistema_critico") and caso.get("impacto_usuarios", 0) > 100:
        decision = "escalar_nivel2"
        escala = False
    elif caso.get("sistema_critico") or caso.get("impacto_usuarios", 0) > 20:
        decision = "escalar_nivel2"
        escala = False
    else:
        decision = "resolver_automatico"
        escala = False
    
    t_fin = time.time()
    return {
        "caso_id": caso["id"], "nivel": caso["nivel"],
        "decision": decision, "escalo_hil": escala,
        "tiempo": round(t_fin - t_ini, 4),
        "ground_truth": caso["ground_truth"]
    }

import time
print("Ejecutando M1 — Baseline RPA (20 casos)...\n")
resultados_inc_m1 = []
for caso in casos_inc:
    r = procesar_incidente_m1_rpa(caso)
    resultados_inc_m1.append(r)
    gt = r["ground_truth"]["resultado"]
    print(f"  {caso['id']} [{caso['nivel']}]: {r['decision']} | GT: {gt} {'✅' if r['decision']==gt else '❌'}")

# Métricas
resueltos = sum(1 for r in resultados_inc_m1 if not r["escalo_hil"] and r["decision"] != "error")
hil_req = [r for r in resultados_inc_m1 if r["ground_truth"].get("requiere_hil")]
hil_det = [r for r in hil_req if r["escalo_hil"]]
errores = [r for r in resultados_inc_m1 if r["decision"] != r["ground_truth"].get("resultado","")]

ARR = round(resueltos / 20 * 100, 1)
HDA = round(len(hil_det) / len(hil_req) * 100, 1) if hil_req else 0
DER = round(len(errores) / 20 * 100, 1)
PT = round(sum(r["tiempo"] for r in resultados_inc_m1) / 20, 4)

print(f"\n{'='*40}")
print(f"M1 — Baseline RPA (Incident Management)")
print(f"ARR = {ARR}% | HDA = {HDA}% | DER = {DER}% | PT = {PT}s")
print(f"{'='*40}")

os.makedirs(r"C:\Users\edurn\practicas projener\resultados_incident", exist_ok=True)
with open(r"C:\Users\edurn\practicas projener\resultados_incident\resultados_inc_m1.json", "w", encoding="utf-8") as f:
    json.dump({"modelo": "M1_RPA_incident", "metricas": {"ARR": ARR, "HDA": HDA, "DER": DER, "PT": PT}, "resultados_por_caso": resultados_inc_m1}, f, ensure_ascii=False, indent=2)
print("✅ Guardado en resultados_inc_m1.json")

Ejecutando M1 — Baseline RPA (20 casos)...

  C01 [Simple]: resolver_automatico | GT: resolver_automatico ✅
  C02 [Simple]: resolver_automatico | GT: resolver_automatico ✅
  C03 [Simple]: resolver_automatico | GT: resolver_automatico ✅
  C04 [Simple]: resolver_automatico | GT: resolver_automatico ✅
  C05 [Simple]: resolver_automatico | GT: resolver_automatico ✅
  C06 [Simple]: resolver_automatico | GT: resolver_automatico ✅
  C07 [Simple]: resolver_automatico | GT: resolver_automatico ✅
  C08 [Simple]: resolver_automatico | GT: resolver_automatico ✅
  C09 [Medio]: escalar_nivel2 | GT: escalar_nivel2 ✅
  C10 [Medio]: escalar_nivel2 | GT: escalar_nivel2 ✅
  C11 [Medio]: escalar_nivel2 | GT: escalar_nivel2 ✅
  C12 [Medio]: escalar_nivel2 | GT: escalar_nivel2 ✅
  C13 [Medio]: escalar_nivel2 | GT: escalar_nivel2 ✅
  C14 [Medio]: escalar_nivel2 | GT: escalar_nivel2 ✅
  C15 [Complejo]: escalar_critico | GT: escalar_critico ✅
  C16 [Complejo]: escalar_nivel2 | GT: escalar_critico ❌
  C17 [Comp

**Resultados M1** — patrón idéntico al Experimento 01:
- ARR=100% — el RPA siempre decide, nunca escala al humano
- HDA=0% — no detecta ninguno de los 6 casos HiL
- DER=15% — 3 errores en casos Complejo (C16, C18, C20): clasifica como `escalar_nivel2` cuando debería ser `escalar_critico`

## M2 — Single Agent (llama-3.1-8b-instant)

In [17]:
# ============================================================
# M2 — SINGLE AGENT LLM (Incident Management)
# llama-3.1-8b-instant, 200 tokens
# ============================================================

from crewai import Agent, Task, Crew, LLM

MODELO_8B = "groq/llama-3.1-8b-instant"
MODELO_70B = "groq/llama-3.3-70b-versatile"
llm_8b = LLM(model=MODELO_8B, temperature=0.0)

def procesar_incidente_m2(caso):
    info_tecnica = get_incidente_info(caso["id"], caso.get("categoria","hardware"), caso.get("sistema_afectado",""))
    info_impacto = verificar_impacto_operativo(caso.get("departamento","IT"), caso.get("impacto_usuarios",1), caso.get("sistema_critico",False))
    info_seguridad = verificar_riesgo_seguridad(caso.get("incidente_seguridad",False), caso.get("datos_en_riesgo",False), caso.get("categoria",""))

    agente = Agent(
        role="Gestor de Incidencias IT",
        goal="Analizar y resolver incidencias IT de ACME Industrial.",
        backstory=f"""Incidente: {caso['descripcion']}
Info técnica: {info_tecnica}
Impacto: {info_impacto}
Seguridad: {info_seguridad}
Responde SOLO con JSON: {{"decision": "resolver_automatico"|"escalar_nivel2"|"escalar_critico", "razon": "texto", "escala_hil": true|false}}""",
        llm=llm_8b, verbose=False
    )

    tarea = Task(
        description=f"Analiza este incidente y decide: {json.dumps(caso, ensure_ascii=False)[:300]}",
        agent=agente,
        expected_output='JSON con decision, razon, escala_hil'
    )

    crew = Crew(agents=[agente], tasks=[tarea], verbose=False)

    try:
        t_ini = time.time()
        resultado = crew.kickoff()
        t_fin = time.time()
        texto = str(resultado)

        decision = "error"
        escala = False
        try:
            match = re.search(r'\{[^{}]*"decision"[^{}]*\}', texto, re.DOTALL)
            if match:
                datos = json.loads(match.group())
                decision = datos.get("decision", "error")
                escala = datos.get("escala_hil", False)
        except:
            pass
        if decision == "error":
            texto_lower = texto.lower()
            if "escalar_critico" in texto_lower or "critico" in texto_lower:
                decision = "escalar_critico"
            elif "escalar_nivel2" in texto_lower or "nivel2" in texto_lower or "nivel 2" in texto_lower:
                decision = "escalar_nivel2"
            elif "resolver" in texto_lower:
                decision = "resolver_automatico"
            else:
                decision = "escalar_nivel2"

        return {
            "caso_id": caso["id"], "nivel": caso["nivel"],
            "decision": decision, "escalo_hil": escala,
            "tiempo": round(t_fin - t_ini, 2),
            "ground_truth": caso["ground_truth"]
        }
    except Exception as e:
        print(f"\n    Error: {str(e)[:100]}")
        return {
            "caso_id": caso["id"], "nivel": caso["nivel"],
            "decision": "error", "escalo_hil": False,
            "tiempo": 0, "ground_truth": caso["ground_truth"], "error": str(e)
        }

print("Ejecutando M2 — Single Agent (20 casos)...\n")
resultados_inc_m2 = []
for i, caso in enumerate(casos_inc):
    print(f"  {caso['id']}...", end=" ")
    for intento in range(3):
        r = procesar_incidente_m2(caso)
        if r["decision"] != "error":
            break
        espera = 60 * (intento + 1)
        print(f"\n    Reintento {intento+1} — esperando {espera}s...", end=" ")
        time.sleep(espera)
    resultados_inc_m2.append(r)
    print(f"{r['decision']} ({'HiL' if r['escalo_hil'] else 'auto'})")
    time.sleep(60)

resueltos = sum(1 for r in resultados_inc_m2 if not r["escalo_hil"] and r["decision"] != "error")
hil_req = [r for r in resultados_inc_m2 if r["ground_truth"].get("requiere_hil")]
hil_det = [r for r in hil_req if r["escalo_hil"]]
errores = [r for r in resultados_inc_m2 if r["decision"] != r["ground_truth"].get("resultado","") and not r["escalo_hil"]]

ARR = round(resueltos / 20 * 100, 1)
HDA = round(len(hil_det) / len(hil_req) * 100, 1) if hil_req else 0
DER = round(len(errores) / 20 * 100, 1)
PT = round(sum(r["tiempo"] for r in resultados_inc_m2) / 20, 2)

print(f"\n{'='*40}")
print(f"M2 — Single Agent (Incident Management)")
print(f"ARR = {ARR}% | HDA = {HDA}% | DER = {DER}% | PT = {PT}s")
print(f"{'='*40}")

with open(r"C:\Users\edurn\practicas projener\resultados_incident\resultados_inc_m2.json", "w", encoding="utf-8") as f:
    json.dump({"modelo": "M2_single_incident", "metricas": {"ARR": ARR, "HDA": HDA, "DER": DER, "PT": PT}, "resultados_por_caso": resultados_inc_m2}, f, ensure_ascii=False, indent=2)
print("✅ Guardado en resultados_inc_m2.json")

Ejecutando M2 — Single Agent (20 casos)...

  C01... 

resolver_automatico (auto)


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

  C02... 

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

escalar_critico (HiL)


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

  C20... 

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

escalar_critico (HiL)

M2 — Single Agent (Incident Management)
ARR = 40.0% | HDA = 100.0% | DER = 0.0% | PT = 0.54s
✅ Guardado en resultados_inc_m2.json


**Resultados M2** — tres decisiones posibles: `resolver_automatico`, `escalar_nivel2`, `escalar_critico`

- ARR=40% — resuelve solo los casos simples (C01-C08)
- HDA=100% — detecta los 6 casos HiL perfectamente
- DER=0% — cero errores de decisión

HDA=100% contrasta con el 30.8% máximo en procurement. Las señales de riesgo en incident management son explícitas y cuantificables (`datos_en_riesgo=True`, `sistema_critico=True`) — el agente las detecta sin ambigüedad.

## M4 — Multi-Agent Mixto (8b×4 + 70b×1)

In [18]:
# ============================================================
# M4 — MULTI-AGENT MIXTO (Incident Management)
# llama-3.1-8b × 4 + llama-3.3-70b × 1 (Resolver)
# ============================================================

llm_70b = LLM(model=MODELO_70B, temperature=0.0)

def procesar_incidente_m4(caso):
    info_tecnica = get_incidente_info(caso["id"], caso.get("categoria","hardware"), caso.get("sistema_afectado",""))
    info_impacto = verificar_impacto_operativo(caso.get("departamento","IT"), caso.get("impacto_usuarios",1), caso.get("sistema_critico",False))
    info_seguridad = verificar_riesgo_seguridad(caso.get("incidente_seguridad",False), caso.get("datos_en_riesgo",False), caso.get("categoria",""))
    ticket = registrar_incidente(caso)

    reporter = Agent(
        role="Gestor de Incidencias",
        goal="Registrar y estructurar el incidente.",
        backstory=f"Ticket registrado: {ticket}. Incidente: {caso['descripcion']}",
        llm=llm_8b, verbose=False
    )
    technical = Agent(
        role="Técnico IT",
        goal="Diagnosticar el problema técnico.",
        backstory=f"Info técnica: {info_tecnica}",
        llm=llm_8b, verbose=False
    )
    security = Agent(
        role="Analista de Seguridad",
        goal="Evaluar el riesgo de seguridad del incidente.",
        backstory=f"Riesgo seguridad: {info_seguridad}",
        llm=llm_8b, verbose=False
    )
    operations = Agent(
        role="Responsable de Operaciones",
        goal="Evaluar el impacto operativo.",
        backstory=f"Impacto operativo: {info_impacto}",
        llm=llm_8b, verbose=False
    )
    resolver = Agent(
        role="Gestor Senior de Incidencias",
        goal="Tomar la decisión final sobre el incidente.",
        backstory=f"""Eres el decisor final. Tienes toda la información:
Incidente: {caso['descripcion']}
Técnico: {info_tecnica}
Seguridad: {info_seguridad}
Operaciones: {info_impacto}
Responde SOLO con JSON: {{"decision": "resolver_automatico"|"escalar_nivel2"|"escalar_critico", "razon": "texto", "escala_hil": true|false}}
IMPORTANTE: escala_hil=true SOLO para escalar_critico.""",
        llm=llm_70b, verbose=False
    )

    t1 = Task(description=f"Registra este incidente: {caso['descripcion']}", agent=reporter, expected_output="Ticket registrado")
    t2 = Task(description="Diagnóstico técnico del incidente.", agent=technical, expected_output="Diagnóstico técnico")
    t3 = Task(description="Evaluación de riesgo de seguridad.", agent=security, expected_output="Evaluación seguridad")
    t4 = Task(description="Evaluación de impacto operativo.", agent=operations, expected_output="Evaluación operativa")
    t5 = Task(description="Decide en JSON la acción a tomar.", agent=resolver, expected_output="JSON con decision, razon, escala_hil")

    crew = Crew(agents=[reporter, technical, security, operations, resolver], tasks=[t1, t2, t3, t4, t5], verbose=False)

    try:
        t_ini = time.time()
        resultado = crew.kickoff()
        t_fin = time.time()
        texto = str(resultado)

        decision = "error"
        escala = False
        try:
            match = re.search(r'\{[^{}]*"decision"[^{}]*\}', texto, re.DOTALL)
            if match:
                datos = json.loads(match.group())
                decision = datos.get("decision", "error")
                escala = datos.get("escala_hil", False)
        except:
            pass
        if decision == "error":
            texto_lower = texto.lower()
            if "escalar_critico" in texto_lower or "critico" in texto_lower:
                decision = "escalar_critico"
                escala = True
            elif "escalar_nivel2" in texto_lower or "nivel2" in texto_lower or "nivel 2" in texto_lower:
                decision = "escalar_nivel2"
            elif "resolver" in texto_lower:
                decision = "resolver_automatico"
            else:
                decision = "escalar_nivel2"

        return {
            "caso_id": caso["id"], "nivel": caso["nivel"],
            "decision": decision, "escalo_hil": escala,
            "tiempo": round(t_fin - t_ini, 2),
            "ground_truth": caso["ground_truth"]
        }
    except Exception as e:
        print(f"\n    Error: {str(e)[:100]}")
        return {
            "caso_id": caso["id"], "nivel": caso["nivel"],
            "decision": "error", "escalo_hil": False,
            "tiempo": 0, "ground_truth": caso["ground_truth"], "error": str(e)
        }

print("Ejecutando M4 — Multi-Agent mixto (20 casos)...\n")
resultados_inc_m4 = []
for i, caso in enumerate(casos_inc):
    print(f"  {caso['id']}...", end=" ")
    for intento in range(3):
        r = procesar_incidente_m4(caso)
        if r["decision"] != "error":
            break
        espera = 70 * (intento + 1)
        print(f"\n    Reintento {intento+1} — esperando {espera}s...", end=" ")
        time.sleep(espera)
    resultados_inc_m4.append(r)
    print(f"{r['decision']} ({'HiL' if r['escalo_hil'] else 'auto'})")
    time.sleep(70)

resueltos = sum(1 for r in resultados_inc_m4 if not r["escalo_hil"] and r["decision"] != "error")
hil_req = [r for r in resultados_inc_m4 if r["ground_truth"].get("requiere_hil")]
hil_det = [r for r in hil_req if r["escalo_hil"]]
errores = [r for r in resultados_inc_m4 if r["decision"] != r["ground_truth"].get("resultado","") and not r["escalo_hil"]]

ARR = round(resueltos / 20 * 100, 1)
HDA = round(len(hil_det) / len(hil_req) * 100, 1) if hil_req else 0
DER = round(len(errores) / 20 * 100, 1)
PT = round(sum(r["tiempo"] for r in resultados_inc_m4) / 20, 2)

print(f"\n{'='*40}")
print(f"M4 — Multi-Agent mixto (Incident Management)")
print(f"ARR = {ARR}% | HDA = {HDA}% | DER = {DER}% | PT = {PT}s")
print(f"{'='*40}")

with open(r"C:\Users\edurn\practicas projener\resultados_incident\resultados_inc_m4.json", "w", encoding="utf-8") as f:
    json.dump({"modelo": "M4_mixto_incident", "metricas": {"ARR": ARR, "HDA": HDA, "DER": DER, "PT": PT}, "resultados_por_caso": resultados_inc_m4}, f, ensure_ascii=False, indent=2)
print("✅ Guardado en resultados_inc_m4.json")

Ejecutando M4 — Multi-Agent mixto (20 casos)...

  C01... 

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

resolver_automatico (auto)
  C02... 

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

resolver_automatico (auto)
  C03... 

...

escalar_critico (HiL)


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

  C20... 

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

escalar_critico (HiL)

M4 — Multi-Agent mixto (Incident Management)
ARR = 45.0% | HDA = 100.0% | DER = 0.0% | PT = 5.04s
✅ Guardado en resultados_inc_m4.json


**Resultados M4:**
- ARR=45% — ligeramente superior a M2 (40%)
- HDA=100% — igual que M2, detecta los 6 casos HiL
- DER=0% — cero errores

## Resumen — Experimento 02

| Modelo | ARR | HDA | DER | PT (s) |
|--------|-----|-----|-----|--------|
| M1 — RPA baseline | 100% | 0% | 15% | ~0 |
| M2 — Single Agent 8b | 40% | 100% | 0% | 0.54 |
| M4 — Multi-Agent mixto | 45% | 100% | 0% | 5.04 |

**Hallazgo clave:** HDA=100% con M2 y M4 confirma que el problema de HDA en procurement es de dominio, no de arquitectura. Las señales explícitas permiten detección perfecta independientemente del modelo.